# Kapitel 7: Skapa chattapplikationer
## Snabbstart för Microsoft Foundry Models API

Denna anteckningsbok är anpassad från [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) som innehåller anteckningsböcker som använder [Azure OpenAI](notebook-azure-openai.ipynb)-tjänster.

> **Obs:** GitHub Models avvecklas i slutet av juli 2026. Denna anteckningsbok riktar sig nu mot [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), som erbjuder samma modellkatalog med möjlighet att prova gratis och erfarenhet av Azure AI Inference SDK.


# Översikt  
"Stora språkmodeller är funktioner som kartlägger text till text. Givet en inmatningssträng av text försöker en stor språkmodell förutsäga vilken text som kommer härnäst"(1). Denna "quickstart"-anteckningsbok kommer att introducera användare till övergripande LLM-koncept, kärnkrav för paket för att komma igång med AML, en mjuk introduktion till promptdesign och flera korta exempel på olika användningsfall. 


## Innehållsförteckning  

[Översikt](#overview)  
[Hur man använder OpenAI Service](#how-to-use-openai-service)  
[1. Skapa din OpenAI-tjänst](#1.-creating-your-openai-service)  
[2. Installation](#2.-installation)    
[3. Referenser](#3.-credentials)  

[Användningsområden](#use-cases)    
[1. Sammanfatta text](#1.-summarize-text)  
[2. Klassificera text](#2.-classify-text)  
[3. Generera nya produktnamn](#3.-generate-new-product-names)  
[4. Finjustera en klassificerare](#4.fine-tune-a-classifier)  

[Referenser](#references)


### Skapa din första prompt  
Denna korta övning ger en grundläggande introduktion till att skicka prompts till en modell i Microsoft Foundry Models för en enkel uppgift "summering".


**Steg**:  
1. Installera `azure-ai-inference` biblioteket i din pythonmiljö, om du inte redan gjort det.  
2. Ladda standardbibliotek för hjälp och konfigurera dina Microsoft Foundry Models-referenser.  
3. Välj en modell för din uppgift  
4. Skapa en enkel prompt för modellen  
5. Skicka din förfrågan till modell-API:t!


### 1. Installera `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Importera hjälpbibliotek och instansiera referenser


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Att hitta rätt modell  
Modeller som GPT-4o och GPT-4o mini kan förstå och generera naturligt språk, och finns tillgängliga i Microsoft Foundry Models-katalogen tillsammans med modeller från Meta, Mistral, Cohere och Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Promptdesign  

"Magin med stora språkmodeller är att genom att tränas för att minimera detta prediktionsfel över enorma mängder text, lär sig modellerna koncept som är användbara för dessa förutsägelser. Till exempel lär de sig koncept som"(1):

* hur man stavar
* hur grammatik fungerar
* hur man omformulerar
* hur man besvarar frågor
* hur man för en konversation
* hur man skriver på många språk
* hur man kodar
* etc.

#### Hur man styr en stor språkmodell  
"Av alla indata till en stor språkmodell är textprompten(1) utan tvekan den mest inflytelserika.

Stora språkmodeller kan promptas för att producera output på några olika sätt:

Instruktion: Säg åt modellen vad du vill ha
Färdigställande: Få modellen att färdigställa början på det du vill ha
Demonstration: Visa modellen vad du vill ha, med antingen:
Några exempel i prompten
Många hundratals eller tusentals exempel i en finjusterings-träningsdataset"



#### Det finns tre grundläggande riktlinjer för att skapa prompts:

**Visa och berätta**. Gör det tydligt vad du vill ha antingen genom instruktioner, exempel, eller en kombination av båda. Om du vill att modellen ska rangordna en lista med objekt i alfabetisk ordning eller klassificera ett stycke efter känslor, visa att det är vad du vill.

**Tillhandahåll kvalitetsdata**. Om du försöker bygga en klassificerare eller få modellen att följa ett mönster, se till att det finns tillräckligt med exempel. Var noga med att korrekturläsa dina exempel — modellen är vanligtvis tillräckligt smart för att se igenom grundläggande stavfel och ge dig ett svar, men den kan också anta att detta är avsiktligt vilket kan påverka svaret.

**Kontrollera dina inställningar.** Inställningarna temperatur och top_p styr hur deterministisk modellen är när den genererar ett svar. Om du frågar efter ett svar där det bara finns ett rätt svar vill du ha dessa inställda lågt. Om du vill ha mer varierade svar vill du kanske ha dem högre. Det vanligaste misstaget folk gör med dessa inställningar är att tro att de är kontroller för "intelligens" eller "kreativitet".


Källa: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Skicka in!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Upprepa samma anrop, hur jämförs resultaten?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Sammanfatta text  
#### Utmaning  
Sammanfatta text genom att lägga till ett 'tl;dr:' i slutet av en textpassage. Notera hur modellen förstår hur man utför ett antal uppgifter utan ytterligare instruktioner. Du kan experimentera med mer beskrivande prompts än tl;dr för att modifiera modellens beteende och anpassa sammanfattningen du får(3).  

Senaste arbeten har visat betydande framsteg i många NLP-uppgifter och benchmarks genom förtränade modeller på stora textkorpusar följt av finjustering på en specifik uppgift. Även om de vanligtvis är uppgiftsagnostiska i arkitektur kräver denna metod ändå uppgiftsspecifika finjusteringsdatamängder med tusentals eller tiotusentals exempel. Däremot kan människor i allmänhet utföra en ny språkuppgift med bara några få exempel eller enkla instruktioner – något som nuvarande NLP-system fortfarande till stor del har svårt att göra. Här visar vi att skalning av språkmodeller kraftigt förbättrar uppgiftsagnostisk, få-exempel-prestanda, ibland till och med uppnår konkurrenskraft med tidigare state-of-the-art-metoder för finjustering.  



Tl;dr


# Övningar för flera användningsfall  
1. Sammanfatta text  
2. Kategorisera text  
3. Generera nya produktnamn


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Klassificera text  
#### Utmaning  
Klassificera objekt i kategorier som tillhandahålls vid inferenstid. I följande exempel tillhandahåller vi både kategorierna och texten som ska klassificeras i prompten(*playground_reference). 

Kundförfrågan: Hej, en av tangenterna på mitt bärbara tangentbord gick nyligen sönder och jag behöver en ersättning:

Klassificerad kategori:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Generera nya produktnamn
#### Utmaning
Skapa produktnamn från exempelord. Här inkluderar vi i prompten information om produkten vi ska generera namn till. Vi tillhandahåller också ett liknande exempel för att visa mönstret vi vill få. Vi har också satt temperaturen högt för att öka slumpmässigheten och få mer innovativa svar.

Produktbeskrivning: En hemmashaker för milkshakes
Fröord: snabb, hälsosam, kompakt.
Produktnamn: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Produktbeskrivning: Ett par skor som passar alla fotstorlekar.
Fröord: anpassningsbar, passform, omni-passform.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referenser  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Bästa praxis för finjustering av GPT-modeller för att klassificera text](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# För mer hjälp  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Bidragsgivare
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
